In [ ]:
def apply_transfer_learning_to_features(features_file, model, out_dir):
    import pandas as pd

    pandas_dataset = pd.read_csv("curated_dataset.csv")
    pandas_dataset.drop_duplicates(subset="Enzyme ID", inplace=True)

    from plants_sm.data_structures.dataset.single_input_dataset import SingleInputDataset

    dataset = SingleInputDataset(dataframe=pandas_dataset, representation_field="Sequence", instances_ids_field="Enzyme ID")

    dataset.load_features(features_file, "proteins")
    dataset._features["place_holder"] = dataset._features["proteins"]
    embedding = model.get_embeddings(dataset)
    from plants_sm.io.pickle import write_pickle
    import os

    features = {"proteins": {}}

    for ids, emb in zip(dataset.identifiers, embedding):
        features["proteins"][ids] = emb

    os.makedirs(out_dir, exist_ok=True)
    write_pickle(os.path.join(out_dir, "features.pkl"), features)


In [ ]:
from enzyme_substrate_prediction.models.protein_model import ESM2_3B
from plants_sm.models.lightning_model import InternalLightningModel
import torch
from torch import nn
from plants_sm.models.fc.fc import DNN
from plants_sm.models.pytorch_model import PyTorchModel

protein_model_ = torch.load("esm2_3b.pt", map_location="cpu")
protein_model = DNN(2560, [2560], 5743, batch_norm=True, last_sigmoid=True)

protein_model.load_state_dict(protein_model_)
model = PyTorchModel(model=protein_model, loss_function=nn.BCELoss, model_name="ec_number", device="cuda:0")

apply_transfer_learning_to_features("features_proteins_esm2_3b", model, "features_proteins_esm2_3b_ec_number_embedding")

In [6]:
from enzyme_substrate_prediction.models.protein_model import ESM2_3B
from plants_sm.models.lightning_model import InternalLightningModel
import torch
from torch import nn
from plants_sm.models.fc.fc import DNN
from plants_sm.models.pytorch_model import PyTorchModel

protein_model_ = torch.load("prot_bert.pt", map_location="cpu")
protein_model = DNN(1024, [2560], 5743, batch_norm=True, last_sigmoid=True)

protein_model.load_state_dict(protein_model_)
model = PyTorchModel(model=protein_model, loss_function=nn.BCELoss, model_name="ec_number", device="cuda:3")

apply_transfer_learning_to_features("features_proteins_prot_bert", model, "prot_bert_ec_number_embedding")

In [4]:
from enzyme_substrate_prediction.models.protein_model import ESM2_3B
from plants_sm.models.lightning_model import InternalLightningModel
import torch
from torch import nn
from plants_sm.models.fc.fc import DNN
from plants_sm.models.pytorch_model import PyTorchModel

protein_model_ = torch.load("esm1b.pt", map_location="cpu")
protein_model = DNN(1280, [2560, 5120], 5743, batch_norm=True, last_sigmoid=True)

protein_model.load_state_dict(protein_model_)
model = PyTorchModel(model=protein_model, loss_function=nn.BCELoss, model_name="ec_number", device="cuda:3")

apply_transfer_learning_to_features("features_proteins_esm1b", model, "esm1b_ec_number_embedding")

In [1]:
from plants_sm.io.pickle import read_pickle

splits = read_pickle("compounds_split/splits_compounds_08.pkl")

import pandas as pd
dataset = pd.read_csv("curated_dataset.csv")

train_ids, val_ids, test_ids  = splits[0]

train_dataset = dataset[dataset["Substrate ID"].isin(train_ids)]
val_dataset = dataset[dataset["Substrate ID"].isin(val_ids)]
test_dataset = dataset[dataset["Substrate ID"].isin(test_ids)]

train_dataset = pd.concat([train_dataset, val_dataset])
train_dataset.to_csv("train_dataset_80_similarity_threshold.csv", index=False)

test_dataset.to_csv("test_dataset_80_similarity_threshold.csv", index=False)